###Streaming Ingestion###

In [0]:
entities=['customers','drivers','locations','payments','trips','vehicles']

In [0]:

import pyspark.sql.functions as F


for entity in entities:

    df_batch= spark.read.format('csv')\
        .option('header','true')\
        .option('inferSchema','true')\
        .load(f'/Volumes/puspark_dbt/source/source_data/{entity}/')

    df_schema= df_batch.schema

    df= spark.readStream.format('csv')\
        .option('header',True)\
        .schema(df_schema)\
        .load(f'/Volumes/puspark_dbt/source/source_data/{entity}/')\
        .withColumn("_source_file", F.col("_metadata.file_path"))\
        .withColumn("_ingested_at", F.current_timestamp())

    df.writeStream.format('delta')\
        .outputMode('append')\
        .option('checkpointLocation',f'/Volumes/puspark_dbt/bronze/checkpoint/{entity}')\
        .trigger(availableNow=True)\
        .toTable(f'puspark_dbt.bronze.{entity}')
    


###updated customers after ingestion of 68 new records###

In [0]:
%sql 
select count(*) from puspark_dbt.bronze.customers;

count(*)
268
